# Step 02 - Prepare Grip Indices

Generate bottom/top grip indices from `raw_structure.xyz`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
from ase.io import read


def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "main.py").exists() and (candidate / "al.gga.psp").exists():
            return candidate
    raise RuntimeError("Project root not found (expected main.py and al.gga.psp).")


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from auto_prep import get_grip_indices

CASE_NAME = "nc_large_vac01"
CASE_DIR = ROOT / "cases" / CASE_NAME
INPUTS_DIR = CASE_DIR / "inputs"

RAW_XYZ = INPUTS_DIR / "raw_structure.xyz"
assert RAW_XYZ.exists(), f"Missing: {RAW_XYZ}"

# Grip controls
END_FRAC = 0.10
MIN_THICKNESS = 2.0
MAX_THICKNESS = 8.0
MIN_LAYERS = 2
MAX_LAYERS = 0  # 0 means unlimited
EPS = 1e-3


In [ ]:
atoms = read(str(RAW_XYZ))
max_layers = None if MAX_LAYERS == 0 else int(MAX_LAYERS)

bottom_idx, top_idx = get_grip_indices(
    atoms,
    end_frac=float(END_FRAC),
    min_thickness=float(MIN_THICKNESS),
    max_thickness=float(MAX_THICKNESS),
    min_layers=int(MIN_LAYERS),
    max_layers=max_layers,
    eps=float(EPS),
    debug=True,
)

np.save(str(INPUTS_DIR / "raw_bottom_idx.npy"), bottom_idx)
np.save(str(INPUTS_DIR / "raw_top_idx.npy"), top_idx)

# Default no-vacancy inputs
np.save(str(INPUTS_DIR / "bottom_idx.npy"), bottom_idx)
np.save(str(INPUTS_DIR / "top_idx.npy"), top_idx)

print("Bottom atoms:", len(bottom_idx))
print("Top atoms   :", len(top_idx))
print("Free atoms  :", len(atoms) - len(np.unique(np.concatenate([bottom_idx, top_idx]))))
